In [ ]:

import os
import deeproot as dr
import math

MODEL_NAME = "telcoChurn"

ACC_THRESH = float(os.getenv("ACC_THRESH", "0.80"))
F1_THRESH  = float(os.getenv("F1_THRESH",  "0.80"))
PER_CLASS_F1_MIN = float(os.getenv("PER_CLASS_F1_MIN", "0.75"))
AUC_MIN    = float(os.getenv("AUC_MIN", "0.80"))

def to_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

metrics = dr.load_metrics(MODEL_NAME)

acc = to_float(metrics.get("accuracy"))
f1_macro = to_float(metrics.get("f1_macro"))
auc = to_float(metrics.get("auc"))

per_class = metrics.get("per_class") or []
f1_vals = []
for row in per_class:
    try:
        v = float(row.get("f1"))
    except (TypeError, ValueError):
        v = None
    if v is not None and not math.isnan(v):
        f1_vals.append(v)

min_f1 = min(f1_vals) if f1_vals else None

reasons = []
passed = True

if f1_macro is None or f1_macro < F1_THRESH:
    passed = False; reasons.append(f"f1_macro {f1_macro:.3f} < {F1_THRESH:.3f}")
if acc is not None and acc < ACC_THRESH:
    passed = False; reasons.append(f"accuracy {acc:.3f} < {ACC_THRESH:.3f}")
if auc is None or auc < AUC_MIN:
    passed = False; reasons.append(f"auc {auc:.3f} < {AUC_MIN:.3f}")
if (min_f1 is not None) and (min_f1 < PER_CLASS_F1_MIN):
    passed = False; reasons.append(f"min per-class F1 {min_f1:.3f} < {PER_CLASS_F1_MIN:.3f}")

print("📊 Métricas:")
print(f"  ACC     : {acc:.4f}")
print(f"  F1-macro: {f1_macro:.4f}")
print(f"  AUC     : {auc:.4f}")
if min_f1 is not None:
    print(f"  Min F1 por classe: {min_f1:.4f}")


print("\nDecisão:", "✅ APROVADO" if passed else "❌ REPROVADO")
if reasons:
    print("Motivo(s):", "; ".join(reasons))
else:
    print("Motivo(s): atende a todos os thresholds.")

if passed:
    dr.register_model(MODEL_NAME, "champion")
    print("✅ Modelo registrado como 'champion'")
else:
    print("❌ Modelo reprovado pelos guardrails")
